In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
import pandas as pd
import os
# we load the CSV file
delivery_path = os.path.join(path, 'Q1_data.csv') #join the kagglehub path with the csv file name
df = pd.read_csv(delivery_path)

print(f"Shape: {df.shape}") #print the shape of the dataframe to make sure things are going well

In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info() #from here we can see that some of the values have null, we will need to work on this to ensure the model is trained correctly

In [ ]:
# Task 4: Write your code here:
df.describe() #we can see from this information that delivery time has a max far from the mean of the rest, just something to keep in mind

In [ ]:
# Task 5: Write your code here:
import matplotlib.pyplot as plt
def check_target_distribution(df, target_column): #we define the function for the plotting
  df[target_column].hist(bins=30, edgecolor='black') #i chose bins to be 30 could go for less but I feel like this is fine

  plt.title(f"Target Distribution ({target_column})")
  plt.xlabel(target_column)
  plt.ylabel("Frequency")
  plt.grid(False)

  plt.show()

check_target_distribution(df, "Delivery_Time")

In [ ]:
# Task 1: Write your code here:
df = df.drop("Order_ID", axis=1) #make sure it's dropping the column with axis=1
df.head()

In [ ]:
# Task 2: Write your code here:
# from the function we defined, we saw that Weather, Traffic_Level, Time_of_Day, Courier_Experience_yrs and Delivery_Time have some missing values, keeping in mind that
# our target is Delivery_Time

def check_missing_values(df):
  missing_values = df.isnull().sum()
  print("Missing Values per Column:")
  print(missing_values[missing_values > 0])
  if missing_values.any():
    print("\nWe need to Handle Missing Values")
  else:
    print("\nNo Missing Values Found.")

check_missing_values(df)
df_clean = df.copy() #making a diff df so that I dont muck things up

# Drop rows where target (Delivery_Time) or key features are missing - can't predict without them
# I didnt drop from Time_of_Day because I don't think it's considered a KEY feature since it doesnt really matter that much tbh when delivering, the other factors do
print(f"Before: {df.shape}")
df_clean = df_clean.dropna(subset=["Weather", "Traffic_Level", "Courier_Experience_yrs", "Delivery_Time"])
print(f"After dropping missing values from Weather, Traffic_Level, Courier_Experience_yrs and Delivery_Time: {df_clean.shape}")

col = "Time_of_Day"
df_clean[col] = df_clean[col].fillna('unknown') #im filling time of day with unknown because I feel like it's not really that important of a feature so entries with that feature missing shouldnt be removed
print("Missing values remaining:", df_clean.isnull().sum().sum())

In [ ]:
# Task 3: Write your code here:
def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df_clean)

In [ ]:
# Task 4: Write your code here:
# I stayed for 30mins tryna get the onehotencoder to work but it just wasted my time imma scrap it pls give me half the bonus or something it wasted my time and I couldnt do all of the last question

from sklearn.preprocessing import LabelEncoder #import LabelEncoder
#Do we have categorical columns?
categorical_cols = df_clean.select_dtypes(include=["object"]).columns

print("Categorical Columns:", list(categorical_cols))
label_encoder = LabelEncoder() # Instantiate Label

for col in categorical_cols:
  df_clean[col] = label_encoder.fit_transform(df_clean[col]) # Apply fit_transform to the copied

df_clean.head()

In [ ]:
# Task 5: Write your code here:
from sklearn.preprocessing import StandardScaler

features = df.columns.drop("Delivery_Time")  # DON'T SCALE THE TARGET

scaler = StandardScaler()
df_clean[features] = scaler.fit_transform(df_clean[features])
df_clean.head()

In [ ]:
# Task 6: Write your code here:
# from the target plot earlier we saw the target is fairly balanced with a slight curve towards the start but its not that big of a deal we wont need to do anything
# For regression, if the target is heavily skewed, we might apply a log transform. However, for this lab we'll work with the raw values.

In [ ]:
# Task 1: Write your code here:
X = df_clean.drop("Delivery_Time", axis=1).astype(float)
y = df_clean['Delivery_Time'].astype(float)

In [ ]:
# Task 2,3,4,5: Write your code here:
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, KFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import mean_absolute_error
import numpy as np

# Stratified split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")

model = RandomForestClassifier(n_estimators=100, max_depth=15,
                               class_weight='balanced', random_state=42)
model.fit(X_train, y_train)
print("Model trained!")

# Predict and evaluate MAE
y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)

print(f"MAE:  ${mae:,.2f}")


#Calculating averaged scores across all folds
kfold = KFold(n_splits=5, shuffle=True, random_state=42)

mae_scores = []
for train_idx, val_idx in kfold.split(X_train):
    X_fold_train, X_fold_val = X_train[train_idx], X_train[val_idx]
    y_fold_train, y_fold_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

    # Train and predict
    model.fit(X_fold_train, y_fold_train)
    y_fold_pred = model.predict(X_fold_val)

    # Calculate metrics
    mae_scores.append(mean_absolute_error(y_fold_val, y_fold_pred))

mae_scores = np.array(mae_scores)

print(f"5-Fold CV Results:")
print(f"MAE:  ${mae_scores.mean():,.2f}")

In [ ]:
# Task 1: Write your code here:
# Feature importance
importance = pd.DataFrame({
    'feature': X.columns,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(importance['feature'], importance['importance'], color='green')
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# Task 2: Write your code here:
pd.DataFrame(y_pred).hist()

In [ ]:
# Task Bonus: Write your code here: